# Notebook 2: Tool Functions & OpenAI Tool Schemas

## Purpose
This notebook defines every tool the LLM agent can call:
- `get_pantry_items` — full inventory with days-to-expiry
- `get_at_risk_items` — items expiring within N days
- `add_pantry_item` — add a newly purchased ingredient
- `remove_pantry_item` — remove a used/discarded item
- `update_quantity` — update remaining quantity after partial use

It also defines the **OpenAI tool schema** (JSON) that tells the LLM what tools exist and how to call them.

> **Prerequisite:** Run `01_database_setup.ipynb` first.

---

## 2.1 Imports and path setup

In [ ]:
import mysql.connector
import json
import os
import sql_openai_config
from datetime import date, datetime
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent

MYSQL_CONFIG = sql_openai_config.get_mysql_config()


def get_connection():
    return mysql.connector.connect(**MYSQL_CONFIG)

print(f"Using database: {MYSQL_CONFIG['database']}")


Using database: smart_pantry


## 2.2 Tool: get_pantry_items

Returns the full pantry inventory sorted by expiry date (soonest first).
The agent must call this before making any recipe suggestion to avoid hallucinating ingredients.

In [12]:
def get_pantry_items() -> list:
    """
    Retrieve all pantry items sorted by expiry date (soonest first).
    Each item includes days_until_expiry computed from today's date.
    Returns a list of dicts.
    """
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT name, category, quantity, unit, expiry_date FROM pantry ORDER BY expiry_date ASC"
    )
    rows = cursor.fetchall()
    conn.close()

    today = date.today()
    result = []
    for name, category, qty, unit, expiry in rows:
        days_left = None
        if expiry:
            days_left = (datetime.strptime(str(expiry), "%Y-%m-%d").date() - today).days
        result.append(
            {
                "name": name,
                "category": category,
                "quantity": qty,
                "unit": unit,
                "expiry_date": str(expiry),
                "days_until_expiry": days_left,
            }
        )
    return result


In [13]:
items = get_pantry_items()
print(len(items))
print(items[0])

175
{'name': 'eggs', 'category': 'dairy', 'quantity': 4.0, 'unit': 'whole', 'expiry_date': '2025-03-28', 'days_until_expiry': -361}


## 2.3 Tool: get_at_risk_items

Filters items expiring within a threshold (default 3 days).
This is the primary tool for zero-waste recipe prioritization.

In [14]:
def get_at_risk_items(threshold_days: int = 3) -> list:
    """
    Return items expiring within threshold_days days.
    Items already expired (negative days) are included with a warning flag.
    """
    all_items   = get_pantry_items()
    at_risk     = []
    for item in all_items:
        d = item['days_until_expiry']
        if d is not None and d <= threshold_days:
            item['status'] = 'EXPIRED' if d < 0 else 'AT RISK'
            at_risk.append(item)
    return at_risk

# Test with default 3-day threshold
at_risk = get_at_risk_items(threshold_days=3)
print(f'Items at risk (<=3 days): {len(at_risk)}')
for item in at_risk:
    print(f"  [{item['status']}] {item['name']:<20} {item['days_until_expiry']} days")

print()
# Test with 7-day threshold
at_risk_7 = get_at_risk_items(threshold_days=7)
print(f'Items at risk (<=7 days): {len(at_risk_7)}')

Items at risk (<=3 days): 6
  [EXPIRED] eggs                 -361 days
  [AT RISK] tomato               3 days
  [AT RISK] spinach fresh        3 days
  [AT RISK] banana               3 days
  [AT RISK] bread whole wheat    3 days
  [AT RISK] croissants           3 days

Items at risk (<=7 days): 25


## 2.4 Tool: add_pantry_item

In [15]:
def add_pantry_item(name: str, category: str, quantity: float,
                    unit: str, expiry_date: str) -> str:
    """
    Add a new ingredient to the pantry database.
    expiry_date must be in ISO format YYYY-MM-DD.
    Returns a confirmation string.
    """
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
    "INSERT INTO pantry (name, category, quantity, unit, expiry_date) VALUES (%s, %s, %s, %s, %s)",
    (name, category, quantity, unit, expiry_date),
)
    conn.commit()
    conn.close()

    return f"Added: {quantity} {unit} of '{name}' (category: {category}, expires: {expiry_date})."

# Test
print(add_pantry_item('avocado', 'fruit', 2, 'whole', '2025-03-28'))

Added: 2 whole of 'avocado' (category: fruit, expires: 2025-03-28).


## 2.5 Tool: remove_pantry_item

In [16]:
def remove_pantry_item(name: str) -> str:
    """
    Remove an item from the pantry by name (case-insensitive).
    Returns a confirmation or not-found message.
    """
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
    "DELETE FROM pantry WHERE LOWER(name) = LOWER(%s)",
    (name,),
)
    affected = cursor.rowcount
    conn.commit()
    conn.close()

    if affected > 0:
        return f"Removed '{name}' from pantry."
    return f"Item '{name}' was not found in the pantry."

# Test
print(remove_pantry_item('avocado'))
print(remove_pantry_item('avocado'))  # should say not found

Removed 'avocado' from pantry.
Item 'avocado' was not found in the pantry.


## 2.6 Tool: update_quantity

In [17]:
def update_quantity(name: str, new_quantity: float) -> str:
    """
    Update the remaining quantity of a pantry item.
    If new_quantity is 0 or less, removes the item automatically.
    """
    if new_quantity <= 0:
        return remove_pantry_item(name)
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
    "UPDATE pantry SET quantity = %s WHERE LOWER(name) = LOWER(%s)",
    (new_quantity, name),
)
    affected = cursor.rowcount
    conn.commit()
    conn.close()

    if affected > 0:
        return f"Updated '{name}' quantity to {new_quantity}."
    return f"Item '{name}' not found in pantry."

# Test
print(update_quantity('eggs', 4))
print(update_quantity('eggs', 0))   # should remove it

# Restore for later use
add_pantry_item('eggs', 'dairy', 4, 'whole', '2025-03-28')
print('Eggs restored.')

Item 'eggs' not found in pantry.
Removed 'eggs' from pantry.
Eggs restored.


## 2.7 OpenAI Tool Schema definitions

These JSON schemas are passed to the `tools` parameter of the OpenAI API.
They tell the LLM: what functions exist, what each one does, and what arguments to pass.

In [18]:
TOOL_DEFINITIONS = [
    {
        'type': 'function',
        'function': {
            'name': 'get_pantry_items',
            'description': (
                'Retrieve all items currently in the pantry, including their quantity, '
                'category, expiry date, and how many days until they expire. '
                'Always call this before suggesting any recipes to avoid hallucinating ingredients.'
            ),
            'parameters': {'type': 'object', 'properties': {}, 'required': []}
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'get_at_risk_items',
            'description': (
                'Get ingredients that are expiring soon (within a given number of days). '
                'Use this to prioritize at-risk ingredients when suggesting recipes.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'threshold_days': {
                        'type': 'integer',
                        'description': 'Number of days to consider at-risk. Defaults to 3.'
                    }
                },
                'required': []
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'add_pantry_item',
            'description': 'Add a new ingredient to the pantry after a shopping trip.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'name'        : {'type': 'string'},
                    'category'    : {'type': 'string', 'description': 'e.g. dairy, vegetable, grain, protein, condiment, spice, fruit'},
                    'quantity'    : {'type': 'number'},
                    'unit'        : {'type': 'string', 'description': 'e.g. grams, cups, whole, ml, liters, pieces'},
                    'expiry_date' : {'type': 'string', 'description': 'ISO format YYYY-MM-DD'}
                },
                'required': ['name', 'category', 'quantity', 'unit', 'expiry_date']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'remove_pantry_item',
            'description': 'Remove an ingredient from the pantry after it has been fully used or discarded.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'name': {'type': 'string', 'description': 'Name of the item to remove'}
                },
                'required': ['name']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'update_quantity',
            'description': 'Update the remaining quantity of a pantry item after it has been partially used in cooking.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'name'         : {'type': 'string'},
                    'new_quantity' : {'type': 'number', 'description': 'New remaining quantity. Pass 0 to remove item.'}
                },
                'required': ['name', 'new_quantity']
            }
        }
    }
]

print(f'Defined {len(TOOL_DEFINITIONS)} tools:')
for t in TOOL_DEFINITIONS:
    print(f"  - {t['function']['name']}")

Defined 5 tools:
  - get_pantry_items
  - get_at_risk_items
  - add_pantry_item
  - remove_pantry_item
  - update_quantity


## 2.8 Tool dispatcher

The dispatcher maps tool names (strings returned by the LLM) to actual Python functions.
The agent loop calls this when the LLM requests a tool.

In [19]:
TOOL_MAP = {
    'get_pantry_items'  : lambda args: get_pantry_items(),
    'get_at_risk_items' : lambda args: get_at_risk_items(**args),
    'add_pantry_item'   : lambda args: add_pantry_item(**args),
    'remove_pantry_item': lambda args: remove_pantry_item(**args),
    'update_quantity'   : lambda args: update_quantity(**args),
}

def dispatch_tool(tool_name: str, tool_args: dict):
    """
    Execute a tool by name with the given arguments.
    Returns the tool result as a JSON-serializable value.
    """
    if tool_name not in TOOL_MAP:
        return f"ERROR: Unknown tool '{tool_name}'"
    return TOOL_MAP[tool_name](tool_args)

# Test the dispatcher
result = dispatch_tool('get_at_risk_items', {'threshold_days': 5})
print(f'Dispatcher test — at-risk items (5 days): {len(result)}')
for r in result:
    print(f"  {r['name']:<20} {r['days_until_expiry']} days")

Dispatcher test — at-risk items (5 days): 19
  eggs                 -361 days
  tomato               3 days
  spinach fresh        3 days
  banana               3 days
  bread whole wheat    3 days
  croissants           3 days
  milk                 4 days
  salmon fillet        4 days
  lettuce romaine      4 days
  mushrooms button     4 days
  raspberries          4 days
  bread sourdough      4 days
  chicken breast       5 days
  kale                 5 days
  mushrooms cremini    5 days
  strawberries         5 days
  burger buns          5 days
  bagels               5 days
  orange juice         5 days
